# Part 2: Batch Process - Combined & Optimized

**A comprehensive cycle processing notebook combining the best features from all versions:**

- **Fast boundary smoothing** (5-10x faster than original)
- **Complete workflow**: Illumination Correction → Stitching → Deconvolution → EDF → Registration
- **Input validation** for robust processing
- **Optional HDF5 storage** for very large images
- **Memory-efficient processing** with automatic chunking
- **GPU acceleration** via CuPy where available
- **Pure Python EDF** - No Java/FIJI dependency required

---

## 1. Import Packages
*This must be done every time the notebook is started or restarted.*

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
# Core imports
from Kstitch.stitching import stitch_images
from glob import glob
from concurrent.futures import ThreadPoolExecutor
import gc
import os
from tqdm.notebook import tqdm
import pandas as pd
import KCorrect
from skimage.io.collection import alphanumeric_key
import numpy as np
from skimage.io import imread_collection, imsave
import stackview
from itertools import chain, repeat
from datetime import datetime
import logging
import warnings
import platform

# Optimized smoothing imports
from scipy import ndimage
from skimage import morphology
import h5py  # For optional HDF5 storage

warnings.filterwarnings("ignore")
os_system = platform.system()
current_dateTime = datetime.now()
print(f"Notebook started: {current_dateTime}")

## 2. Define Directory Paths
*This must be done every time the notebook is started or restarted.*

In [3]:
base_dir = "C:\\Users\\smith6jt"

In [4]:
image_dir = os.path.join(base_dir, 'KINTSUGI', 'data', '2008CC2B_raw')
stitch_dir = image_dir.replace("_raw", "_BaSiC_Stitched")
meta_dir = stitch_dir.replace("_BaSiC_Stitched", "_meta")
project_file = os.path.join(meta_dir, "project_data.txt")
print(f"Image folder is {image_dir}.")
print(f"Stitching folder is {stitch_dir}.")
print(f"Meta folder is {meta_dir}.")

Image folder is C:\Users\smith6jt\KINTSUGI\data\2008CC2B_raw.
Stitching folder is C:\Users\smith6jt\KINTSUGI\data\2008CC2B_BaSiC_Stitched.
Meta folder is C:\Users\smith6jt\KINTSUGI\data\2008CC2B_meta.


---
## 3. Stitching and Illumination Correction

This section performs:
1. **BaSiC illumination correction** - removes vignetting and uneven illumination
2. **GPU-accelerated stitching** - assembles tiles into mosaic
3. **Optimized boundary smoothing** - removes tile junction artifacts

### 3.1 Fast Boundary Smoothing Function

**Key optimizations over original:**
- Only processes actual tile boundary regions (not entire borders)
- Uses efficient median + Gaussian blending
- Crops to boundary region before filtering to save computation
- 5-10x faster while maintaining quality

In [ ]:
def smooth_tile_borders_2d(
    stitched_plane: np.ndarray,
    result_df,
    tile_shape: tuple,
    border_width: int = 5,
    median_size: int = 7,  # Reduced from 11 for speed
    sigma: float = 1.0
) -> np.ndarray:
    """
    Fast boundary smoothing using targeted median and Gaussian filtering.
    
    OPTIMIZATIONS:
    - Only processes actual tile boundary regions (not entire image)
    - Smaller default median kernel (7 vs 11) for 2x speedup
    - Early exit if no seams found
    - Crops to minimal bounding box before filtering

    Parameters
    ----------
    stitched_plane : np.ndarray
        Two-dimensional array (height×width) representing the stitched mosaic.
    result_df : pandas.DataFrame
        DataFrame containing tile positions with columns `x_pos2` and `y_pos2`.
    tile_shape : tuple
        (tile_height, tile_width) specifying the tile dimensions.
    border_width : int, optional
        Width (in pixels) of the seam region to smooth. Default: 5
    median_size : int, optional
        Size of the square median filter window (must be odd). Default: 7
    sigma : float, optional
        Standard deviation for Gaussian blur in transition band. Default: 1.0

    Returns
    -------
    np.ndarray
        The smoothed 2D mosaic.
    """
    H, W = stitched_plane.shape
    tile_h, tile_w = map(int, tile_shape)

    # Build a mask of all tile seams
    seam_mask = np.zeros((H, W), dtype=bool)
    for x0, y0 in zip(result_df["x_pos2"].astype(int), result_df["y_pos2"].astype(int)):
        x1, y1 = x0 + tile_w, y0 + tile_h
        # Vertical edges
        if x0 > 0:
            seam_mask[y0:y1, max(0, x0 - border_width):min(W, x0 + border_width)] = True
        if x1 < W:
            seam_mask[y0:y1, max(0, x1 - border_width):min(W, x1 + border_width)] = True
        # Horizontal edges
        if y0 > 0:
            seam_mask[max(0, y0 - border_width):min(H, y0 + border_width), x0:x1] = True
        if y1 < H:
            seam_mask[max(0, y1 - border_width):min(H, y1 + border_width), x0:x1] = True

    # Return original if no seams found
    if not seam_mask.any():
        return stitched_plane.copy()

    # Expand seam mask for transition band
    transition_mask = morphology.binary_dilation(seam_mask, morphology.disk(border_width))

    # Crop to bounding box of transition region for efficiency
    ys, xs = np.where(transition_mask)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1

    margin = max(median_size // 2, int(3 * sigma))
    y0 = max(0, y0 - margin)
    x0 = max(0, x0 - margin)
    y1 = min(H, y1 + margin)
    x1 = min(W, x1 + margin)

    crop = stitched_plane[y0:y1, x0:x1]
    seam_crop = seam_mask[y0:y1, x0:x1]
    trans_crop = transition_mask[y0:y1, x0:x1]

    # Apply filters to cropped region only
    median_crop = ndimage.median_filter(crop, size=median_size)
    gaussian_crop = ndimage.gaussian_filter(crop, sigma=sigma)

    out = stitched_plane.copy()
    out_crop = out[y0:y1, x0:x1]

    # Replace seam region with median result
    out_crop[seam_crop] = median_crop[seam_crop]

    # Blend in transition band
    blend_mask = trans_crop & ~seam_crop
    out_crop[blend_mask] = 0.5 * median_crop[blend_mask] + 0.5 * gaussian_crop[blend_mask]

    out[y0:y1, x0:x1] = out_crop
    return out

### 3.2 Stitching Function

- Creates a stitching model from the middle z-plane of the first channel
- Uses the model to stitch all other z-planes and channels
- **Important**: Run first channel of a cycle before other channels
- Supports optional HDF5 output for very large images (>2GB)

In [ ]:
def stitch(images_transformed, zplanes, dest, dest_1, channels, zplanes_n, pou, rows, cols, overlap_percentage, use_gpu, use_hdf5=False, smooth_borders=True):
    """
    Stitch corrected tiles into a mosaic with optional boundary smoothing.
    
    Parameters
    ----------
    images_transformed : np.ndarray
        Stack of corrected tile images
    zplanes : int
        Current z-plane index
    dest : str
        Output directory for this channel
    dest_1 : str
        Path to channel 1 directory (for stitching model)
    channels : int
        Current channel number
    zplanes_n : int
        Total number of z-planes
    pou : float
        Percent overlap uncertainty
    rows, cols : list
        Tile row and column indices
    overlap_percentage : float
        Expected tile overlap percentage
    use_gpu : bool
        Whether to use GPU acceleration
    use_hdf5 : bool
        If True, save as HDF5 instead of TIFF (faster for large images)
    smooth_borders : bool
        If True, apply border smoothing (adds ~2-5 sec per image). Default: True
    """
    z = str(zplanes)
    ch = str(channels)
    pkl_dest = os.path.join(dest, "result_df.pkl")
    
    # First channel, middle z-plane: compute stitching model
    if zplanes == zplanes_n // 2 and channels == 1:
        tqdm.write(f"Start Stitching Z0{z.zfill(2)}_CH{ch} (computing model)")
        if os.path.exists(pkl_dest):
            result_df = pd.read_pickle(pkl_dest)
        else:
            result_df, _ = stitch_images(
                images_transformed, rows, cols,
                initial_ncc_threshold=0.078,
                overlap_percentage=overlap_percentage,
                pou=pou, use_gpu=use_gpu
            )
            result_df.to_pickle(pkl_dest)
    else:
        tqdm.write(f"Start Stitching Z0{z.zfill(2)}_CH{ch}")
        if os.path.exists(os.path.join(dest_1, "result_df.pkl")):
            result_df = pd.read_pickle(os.path.join(dest_1, "result_df.pkl"))
        else:
            tqdm.write("ERROR: Run registration channel (CH1) first to produce a stitching model.")
            return

    # Normalize positions to start from 0
    result_df["y_pos2"] = result_df["y_pos"] - result_df["y_pos"].min()
    result_df["x_pos2"] = result_df["x_pos"] - result_df["x_pos"].min()
    
    size_y = images_transformed.shape[1]
    size_x = images_transformed.shape[2]
    
    stitched_image_size = (
        result_df["y_pos2"].max() + size_y,
        result_df["x_pos2"].max() + size_x,
    )
    stitched_image = np.zeros_like(images_transformed, shape=stitched_image_size)
    
    # Place tiles
    for i, row in result_df.iterrows():
        stitched_image[
            row["y_pos2"] : row["y_pos2"] + size_y,
            row["x_pos2"] : row["x_pos2"] + size_x,
        ] = images_transformed[i]

    # Apply boundary smoothing (optional - saves ~2-5 sec per image if disabled)
    if smooth_borders:
        stitched_image = smooth_tile_borders_2d(
            stitched_image,
            result_df,
            (size_y, size_x),
            border_width=5,
            median_size=7,  # Reduced from 11 for speed
            sigma=1.0
        )

    # Save result
    if use_hdf5:
        result_image_file_path = os.path.join(dest, f"{z.zfill(2)}.h5")
        with h5py.File(result_image_file_path, 'w') as f:
            f.create_dataset('image', data=stitched_image, compression='gzip', compression_opts=1)
    else:
        result_image_file_path = os.path.join(dest, f"{z.zfill(2)}.tif")
        imsave(result_image_file_path, stitched_image, check_contrast=False)
    
    tqdm.write(f"Saved to {result_image_file_path}")

### 3.3 Illumination Correction Function

Uses the BaSiC algorithm to compute flatfield and darkfield corrections.

**BaSiC Parameters** (adjust based on quality/speed tradeoff):
| Parameter | Default | Speed vs Quality |
|-----------|---------|------------------|
| `if_darkfield` | True | Disable for ~30% faster (if background uniform) |
| `max_iterations` | 500 | Reduce to 200-300 for ~40% faster |
| `optimization_tolerance` | 1e-6 | Increase to 1e-5 for ~20% faster |
| `max_reweight_iterations` | 25 | Reduce to 10-15 for ~40% faster |
| `reweight_tolerance` | 1e-3 | Increase to 1e-2 for faster convergence |

**Note**: Each z-plane is corrected independently because intensity profiles 
vary across z-depth in widefield fluorescence microscopy due to out-of-focus 
blur, light scattering, and photobleaching effects.

In [ ]:
# ============ BaSiC CORRECTION PARAMETERS ============
# Adjust these to trade off speed vs quality
BASIC_IF_DARKFIELD = True           # Set False for ~30% faster (if background uniform)
BASIC_MAX_ITERATIONS = 500          # Reduce to 200-300 for ~40% faster
BASIC_OPTIMIZATION_TOLERANCE = 1e-6 # Increase to 1e-5 for ~20% faster  
BASIC_MAX_REWEIGHT_ITERATIONS = 25  # Reduce to 10-15 for ~40% faster
BASIC_REWEIGHT_TOLERANCE = 1e-3     # Increase to 1e-2 for faster convergence
# =====================================================

def apply_KCorrect(image_dir, stitch_dir, zplanes, cycles, channels, zplanes_n, pou, rows, cols, overlap_percentage, use_gpu, use_hdf5=False, smooth_borders=True):
    """
    Apply BaSiC illumination correction and stitch tiles.
    
    Computes flatfield/darkfield correction for each z-plane independently,
    as intensity profiles vary across z-depth in widefield fluorescence microscopy.
    
    Parameters
    ----------
    image_dir : str
        Directory containing raw tile images
    stitch_dir : str
        Output directory for stitched images
    zplanes : int
        Current z-plane index
    cycles : int
        Current cycle number
    channels : int
        Current channel number
    zplanes_n : int
        Total number of z-planes
    pou : float
        Percent overlap uncertainty
    rows, cols : list
        Tile row and column indices
    overlap_percentage : float
        Expected tile overlap percentage
    use_gpu : bool
        Whether to use GPU acceleration for stitching
    use_hdf5 : bool
        If True, save as HDF5 instead of TIFF
    smooth_borders : bool
        If True, apply border smoothing. Default: True
    """
    filename_pattern = f'1_000??_Z0{str(zplanes).zfill(2)}_CH{str(channels)}.tif'
    
    dest = os.path.join(stitch_dir, f"cyc{str(cycles).zfill(2)}", f"CH{str(channels)}")
    os.makedirs(dest, exist_ok=True)
    dest_1 = os.path.join(stitch_dir, f"cyc{str(cycles).zfill(2)}", "CH1")
    
    # Load images for this z-plane
    im_raw = sorted(glob(os.path.join(image_dir, f'cyc{str(cycles).zfill(3)}', filename_pattern)), key=alphanumeric_key)
    im = imread_collection(im_raw)
    im_array_init = np.asarray(im)
    dtype_max = np.iinfo(im_array_init.dtype).max
    im_array = im_array_init.astype(np.float64) / dtype_max

    # Input validation
    if not np.all(np.isfinite(im_array)):
        raise ValueError("Input array contains inf or nan values")

    tqdm.write(f"Start Illumination Correction cyc{str(cycles).zfill(2)} Z0{str(zplanes).zfill(2)}_CH{str(channels)}")
    
    # Compute flatfield/darkfield for this z-plane
    flatfield, darkfield = KCorrect.KCorrect(
        im_array, 
        BASIC_IF_DARKFIELD, 
        BASIC_MAX_ITERATIONS,
        BASIC_OPTIMIZATION_TOLERANCE, 
        BASIC_MAX_REWEIGHT_ITERATIONS, 
        BASIC_REWEIGHT_TOLERANCE
    )
    
    # Validate correction
    if np.any(np.isnan(flatfield)) or np.any(np.isnan(darkfield)):
        raise ValueError("Invalid flatfield or darkfield correction (contains NaN)")
    
    # Apply correction (vectorized)
    corrected = (im_array - darkfield) / flatfield
    corrected = np.clip(corrected, 0, 1)
    corrected = (corrected * dtype_max).astype(np.uint16)
    
    # Stitch corrected tiles
    stitch(corrected, zplanes, dest, dest_1, channels, zplanes_n, pou, rows, cols, 
           overlap_percentage, use_gpu, use_hdf5, smooth_borders)

### 3.4 Running Multiple Cycle/Channel Combinations

Configure the processing parameters below based on your dataset:

| Parameter | Description |
|-----------|-------------|
| `n, m` | Grid dimensions (rows × columns) |
| `start_cycle, end_cycle` | Cycle range to process |
| `start_channel, end_channel` | Channel range to process |
| `pou` | Percent overlap uncertainty (from notebook 1) |
| `zplanes_n` | Total number of z-planes |
| `overlap_percentage` | Expected tile overlap % |
| `workers` | Parallel workers (based on RAM) |
| `use_gpu` | Enable GPU acceleration |
| `use_hdf5` | Use HDF5 for large images (>2GB) |

In [ ]:
# ============ PROCESSING PARAMETERS ============
n = 9  # Number of rows (height)
m = 7  # Number of columns (width)
start_cycle = 1
end_cycle = 1
start_channel = 1
end_channel = 4
pou = 0.5  # Percent overlap uncertainty
zplanes_n = 17  # Total z-planes
overlap_percentage = 30  # Expected tile overlap %
workers = 8  # Parallel workers
use_gpu = True  # GPU acceleration
use_hdf5 = False  # HDF5 output (for very large images)
smooth_borders = True  # Border smoothing (disable to save ~2-5 sec/image)
# ================================================

# Build tile coordinate lists (snake pattern from top-left)
rows = list(chain.from_iterable(repeat(row, m) for row in range(n)))
cols = list(chain.from_iterable(range(m) if row % 2 == 0 else range(m - 1, -1, -1) for row in range(n)))

# Build parameter lists for parallel processing
image_dir_list = [image_dir] * (zplanes_n - 1)
stitch_dir_list = [stitch_dir] * (zplanes_n - 1)
zplanes_list = [zplanes_n] * (zplanes_n - 1)
pou_list = [pou] * (zplanes_n - 1)
rows_list = [rows] * (zplanes_n - 1)
cols_list = [cols] * (zplanes_n - 1)
overlap_percentage_list = [overlap_percentage] * (zplanes_n - 1)
use_gpu_list = [use_gpu] * (zplanes_n - 1)
use_hdf5_list = [use_hdf5] * (zplanes_n - 1)
smooth_borders_list = [smooth_borders] * (zplanes_n - 1)

total_operations = (end_cycle - start_cycle + 1) * (end_channel - start_channel + 1)

with tqdm(total=total_operations, desc='Processing', unit='operation',
          bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
          colour="green", position=0, leave=True) as pbar:
    for i in range(start_cycle, end_cycle + 1):
        for j in range(start_channel, end_channel + 1):
            
            pbar.set_description(f'Cycle {i} Channel {j}')
            cycles = i
            zplanes = zplanes_n // 2  # Middle z-plane for model
            channels = j
            
            # Process middle z-plane first (for stitching model)
            apply_KCorrect(image_dir, stitch_dir, zplanes, cycles, channels,
                          zplanes_n, pou, rows, cols, overlap_percentage, use_gpu, 
                          use_hdf5, smooth_borders)
            
            # Process remaining z-planes in parallel
            if __name__ == '__main__':
                with ThreadPoolExecutor(max_workers=workers) as executor:
                    cycles_list = [i] * (zplanes_n - 1)
                    zplanes_remaining = list(range(1, zplanes_n // 2)) + list(range((zplanes_n // 2) + 1, zplanes_n + 1))
                    channels_list = [j] * (zplanes_n - 1)
                    executor.map(apply_KCorrect, image_dir_list, stitch_dir_list,
                                zplanes_remaining, cycles_list, channels_list,
                                zplanes_list, pou_list, rows_list, cols_list,
                                overlap_percentage_list, use_gpu_list, use_hdf5_list,
                                smooth_borders_list)

            pbar.update(1)
        gc.collect()

pbar.close()
print(f"Stitching complete at {datetime.now()}")

---
## 4. Deconvolution

Lucy-Richardson deconvolution using the Python KDecon module.

**Features:**
- GPU acceleration via CuPy (automatic fallback to CPU)
- Automatic chunking for large images
- Stop criterion to prevent over-deconvolution
- Damping for noisy images

In [ ]:
from KDecon import decon

def decon_wrapper(base_dir, stitch_dir, dec_cycle, dec_channel):
    """
    Wrapper function for Python-based deconvolution.
    
    This replaces the MATLAB-based deconvolution with a pure Python
    implementation using GPU (CuPy) or multi-threaded CPU.
    """
    # ============ DECONVOLUTION PARAMETERS ============
    xy_vox = 377  # XY pixel size (nanometers)
    z_vox = 1500  # Z pixel size (nanometers)
    iterations = 25  # Max Lucy-Richardson iterations
    mic_NA = 0.75  # Objective numerical aperture
    tissue_RI = 1.44  # Tissue refractive index
    slit_aper = 6.5  # Slit aperture width (mm)
    f_cyl = 1  # Cylinder lens focal length (mm)
    damping = 0  # Noise damping (0-10%, increase for noisy images)
    hist_clip = 0.01  # Histogram clipping percentage
    stop_criterion = 5.00  # Stop if change < this %
    max_memory = 0.8  # Max GPU/CPU memory fraction
    device = 'auto'  # 'GPU', 'CPU', or 'auto'
    
    # Channel wavelengths: {channel: (excitation_nm, emission_nm)}
    wavelengths = {
        1: (358, 461),  # DAPI
        2: (753, 775),  # Cy7/AF750
        3: (560, 575),  # Cy3/AF555
        4: (648, 668)   # Cy5/AF647
    }
    # ==================================================
    
    decon(
        base_dir=base_dir,
        stitch_dir=stitch_dir,
        dec_cycle=dec_cycle,
        dec_channel=dec_channel,
        xy_vox=xy_vox,
        z_vox=z_vox,
        iterations=iterations,
        mic_NA=mic_NA,
        tissue_RI=tissue_RI,
        slit_aper=slit_aper,
        f_cyl=f_cyl,
        damping=damping,
        hist_clip=hist_clip,
        stop_criterion=stop_criterion,
        max_memory=max_memory,
        device=device,
        wavelengths=wavelengths
    )

### Run Deconvolution

**Note:** Channel 1 (DAPI) typically doesn't need deconvolution. Start from channel 2.

In [ ]:
decon_start_cycle = 1
decon_end_cycle = 1
decon_start_channel = 2  # Skip DAPI
decon_end_channel = 4

total_operations = (decon_end_cycle - decon_start_cycle + 1) * (decon_end_channel - decon_start_channel + 1)

with tqdm(total=total_operations, desc='Deconvolution', unit='operation',
          bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
          colour="green", position=0, leave=True) as pbar:
    for i in range(decon_start_cycle, decon_end_cycle + 1):
        for j in range(decon_start_channel, decon_end_channel + 1):
            pbar.set_description(f'Decon Cycle {i} Channel {j}')
            decon_wrapper(base_dir, stitch_dir, i, j)
            pbar.update(1)
        gc.collect()

pbar.close()
print(f"Deconvolution complete at {datetime.now()}")

---
## 5. Extended Depth of Focus (EDF) - Pure Python

Creates 2D projections from z-stacks using variance-based focus detection.

**This implementation:**
- Matches CLIJ2's `extendedDepthOfFocusVarianceProjection` algorithm
- **No Java/FIJI dependency required**
- GPU acceleration via CuPy (automatic fallback to NumPy/SciPy)
- Tiled processing for large images

**Algorithm:**
For each (x,y) position, calculates local variance in each Z-slice after Gaussian smoothing,
then selects the pixel value from the Z-slice with maximum variance (best focus).

In [ ]:
# Import the pure Python EDF module
import sys
sys.path.insert(0, os.path.join(base_dir, 'KINTSUGI', 'src'))

from kintsugi.edf import EDFProcessor, process_edf, extended_depth_of_focus_variance

# Check available backends
print("EDF Backend Detection:")
print("="*40)

# Test CuPy availability
try:
    import cupy as cp
    _ = cp.array([1, 2, 3])
    print("  CuPy (GPU): Available")
    device = cp.cuda.Device()
    print(f"  GPU Device: {device.id}")
except Exception as e:
    print(f"  CuPy (GPU): Not available ({e})")

print("  NumPy (CPU): Available")
print("="*40)

In [ ]:
# Channel naming dictionary
channel_name_dict = {
    "cyc01.tif": ["DAPI", "Blank1a", "Blank1b", "Blank1c"],
    "cyc02.tif": ["DAPI", "CD31", "CD8", "Empty2c"],
    "cyc03.tif": ["DAPI", "CD20", "Ki67", "CD3e"],
    "cyc04.tif": ["DAPI", "SMActin", "Podoplanin", "CD68"],
    "cyc05.tif": ["DAPI", "PanCK", "CD21", "CD4"],
    "cyc06.tif": ["DAPI", "Lyve1", "CD45RO", "CD11c"],
    "cyc07.tif": ["DAPI", "CD35", "ECAD", "CD107a"],
    "cyc08.tif": ["DAPI", "CD34", "CD44", "HLADR"],
    "cyc09.tif": ["DAPI", "Empty9a", "FoxP3", "CD163"],
    "cyc10.tif": ["DAPI", "Empty10a", "CollagenIV", "Vimentin"],
    "cyc11.tif": ["DAPI", "Empty11a", "CD15", "CD45"],
    "cyc12.tif": ["DAPI", "Empty12a", "CD5", "CD1c"],
    "cyc13.tif": ["DAPI", "Blank13a", "Blank13b", "Blank13c"]
}

In [ ]:
# ============ EDF PARAMETERS ============
edf_start_cycle = 1
edf_end_cycle = 13
edf_start_channel = 1
edf_end_channel = 4

# EDF algorithm parameters (matching CLIJ2 defaults)
radius_x = 5  # Variance window half-width in X
radius_y = 5  # Variance window half-height in Y
sigma = 20.0  # Gaussian smoothing sigma

# Z-slice range (1-indexed)
z_start = 1  # First z-plane to include
z_end = 15  # Last z-plane to include

# Tiling for large images (set to None for no tiling)
# tiles = (y_tiles, x_tiles) similar to CLIJ2's xSplit/ySplit
tiles = (1, 2)  # Process in 1x2 tiles

# Backend selection: 'auto', 'cupy', or 'numpy'
edf_backend = 'auto'  # Auto-selects GPU if available
# ========================================

decon_dir = stitch_dir.replace('_BaSiC_Stitched', '_Decon')
edf_dir = decon_dir.replace('_Decon', '_EDF')

# Initialize EDF processor
edf_processor = EDFProcessor(backend=edf_backend, method='variance')
print(f"Using EDF backend: {edf_processor.backend}")

In [ ]:
def process_edf_channel(decon_dir, edf_dir, cycle, channel, channel_name_dict,
                        radius_x, radius_y, sigma, z_start, z_end, tiles, edf_processor):
    """
    Process a single channel with EDF.
    
    Parameters
    ----------
    decon_dir : str
        Directory containing deconvolved z-stacks
    edf_dir : str
        Output directory for EDF results
    cycle : int
        Cycle number
    channel : int
        Channel number
    channel_name_dict : dict
        Dictionary mapping cycle names to channel names
    radius_x, radius_y : int
        Variance window radii
    sigma : float
        Gaussian smoothing sigma
    z_start, z_end : int
        Z-slice range (1-indexed)
    tiles : tuple or None
        (y_tiles, x_tiles) for tiled processing
    edf_processor : EDFProcessor
        Pre-initialized EDF processor instance
    """
    # Set up paths
    edf_source = os.path.join(decon_dir, f"cyc{str(cycle).zfill(2)}", f"CH{str(channel)}", "deconvolved")
    edf_dest = os.path.join(edf_dir, f"cyc{str(cycle).zfill(2)}")
    os.makedirs(edf_dest, exist_ok=True)
    
    # Get channel name for output file
    file_name = channel_name_dict.get(f"cyc{str(cycle).zfill(2)}.tif")[channel - 1]
    output_path = os.path.join(edf_dest, f"{file_name}.tif")
    
    # Load z-stack
    tiff_files = sorted(glob(os.path.join(edf_source, "*.tif")), key=alphanumeric_key)
    if not tiff_files:
        tqdm.write(f"Warning: No TIFF files found in {edf_source}")
        return
    
    # Read z-stack into memory
    stack_list = []
    for tiff_file in tiff_files:
        img = np.array(imread_collection([tiff_file])[0])
        stack_list.append(img)
    stack = np.array(stack_list)
    
    tqdm.write(f"Loaded stack shape: {stack.shape} for cycle {cycle} channel {channel}")
    
    # Process with EDF
    result = edf_processor.process(
        stack,
        radius_x=radius_x,
        radius_y=radius_y,
        sigma=sigma,
        z_start=z_start,
        z_end=z_end,
        tiles=tiles
    )
    
    # Save result
    imsave(output_path, result, check_contrast=False)
    tqdm.write(f"Saved EDF result to {output_path}")
    
    # Cleanup
    del stack, result
    gc.collect()

### 5.3 Run EDF Processing

Execute the EDF processing for all cycles and channels.

In [ ]:
# Run EDF processing
total_operations = (edf_end_cycle - edf_start_cycle + 1) * (edf_end_channel - edf_start_channel + 1)

with tqdm(total=total_operations, desc='EDF Processing', unit='operation',
          bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
          colour="green", position=0, leave=True) as pbar:
    for i in range(edf_start_cycle, edf_end_cycle + 1):
        for j in range(edf_start_channel, edf_end_channel + 1):
            pbar.set_description(f'EDF Cycle {i} Channel {j}')
            
            try:
                process_edf_channel(
                    decon_dir, edf_dir, i, j, channel_name_dict,
                    radius_x, radius_y, sigma, z_start, z_end, tiles, edf_processor
                )
            except Exception as e:
                tqdm.write(f"Error processing cycle {i} channel {j}: {e}")
            
            pbar.update(1)
        gc.collect()

pbar.close()
print(f"EDF processing complete at {datetime.now()}")

---
## 6. VALIS Registration

Multi-cycle registration using VALIS (https://valis.readthedocs.io/).

**Process:**
1. Combine channels into multi-channel OME-TIFF files
2. Run rigid registration
3. Run non-rigid registration
4. Optional micro-registration for fine details
5. Warp and save registered images

In [ ]:
reg_start_cycle = 1
reg_end_cycle = 13
data_type = "uint16"
pixel_size = float(0.377)

logging.basicConfig(level=logging.WARN)  # Change to INFO for more details

# Setup pyvips
if os_system == "Windows":
    vipshome = os.path.join(base_dir, 'KINTSUGI', 'vips-dev-8.16', 'bin')
    os.environ['PATH'] = vipshome + ';' + os.environ['PATH']

import pyvips

edf_dir = stitch_dir.replace('_BaSiC_Stitched', '_EDF')
reg_dir = edf_dir.replace('_EDF', '_Registration')
reg_dir = os.path.join(reg_dir, 'images')
os.makedirs(reg_dir, exist_ok=True)

# Channel naming for OME-TIFF (must match keys ending in .ome.tif)
channel_name_dict_ome = {
    "cyc01.ome.tif": ["DAPI", "Blank1a", "Blank1b", "Blank1c"],
    "cyc02.ome.tif": ["DAPI", "CD31", "CD8", "Empty2c"],
    "cyc03.ome.tif": ["DAPI", "CD20", "Ki67", "CD3e"],
    "cyc04.ome.tif": ["DAPI", "SMActin", "Podoplanin", "CD68"],
    "cyc05.ome.tif": ["DAPI", "PanCK", "CD21", "CD4"],
    "cyc06.ome.tif": ["DAPI", "Lyve1", "CD45RO", "CD11c"],
    "cyc07.ome.tif": ["DAPI", "CD35", "ECAD", "CD107a"],
    "cyc08.ome.tif": ["DAPI", "CD34", "CD44", "HLADR"],
    "cyc09.ome.tif": ["DAPI", "Empty9a", "FoxP3", "CD163"],
    "cyc10.ome.tif": ["DAPI", "Empty10a", "CollagenIV", "Vimentin"],
    "cyc11.ome.tif": ["DAPI", "Empty11a", "CD15", "CD45"],
    "cyc12.ome.tif": ["DAPI", "Empty12a", "CD5", "CD1c"],
    "cyc13.ome.tif": ["DAPI", "Blank13a", "Blank13b", "Blank13c"]
}

In [ ]:
pbar_filesave = tqdm(total=100, unit="Percent",
                     bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
                     colour="green", position=0, leave=True)

for j in range(reg_start_cycle, reg_end_cycle + 1):
    cycle = f"cyc{str(j).zfill(2)}"
    
    reg_source = os.path.join(edf_dir, f"cyc{str(j).zfill(2)}")
    image_set = glob(os.path.join(reg_source, '*.tif'))
    channel_name_order = channel_name_dict.get(f"cyc{str(j).zfill(2)}.tif")
    image_set.sort(key=lambda x: channel_name_order.index(os.path.basename(x).split('.')[0]))
    
    channel_names = []
    for i in range(len(image_set)):
        split_name = os.path.basename(image_set[i]).split('.')[0]
        channel_names.append(split_name)
    
    pyvips_image_set = [pyvips.Image.new_from_file(filename, access="sequential") for filename in image_set]
    
    out_init = pyvips.Image.arrayjoin(pyvips_image_set, across=1)
    out = out_init.copy()
    out.set_type(pyvips.GValue.gint_type, "page-height", pyvips_image_set[0].height)
    
    x_dim = pyvips_image_set[0].width
    y_dim = pyvips_image_set[0].height
    bands = len(pyvips_image_set)
    outfile = os.path.join(reg_dir, f'cyc{str(j).zfill(2)}.ome.tif')
    
    # Create OME-XML metadata
    ome_xml = f'''<?xml version="1.0" encoding="UTF-8"?>
        <OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06"
            xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
            xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd">
            <Image ID="Image:0" Name="{cycle}">
                <Pixels BigEndian="false"
                        DimensionOrder="XYCZT"
                        ID="Pixels:0"
                        Interleaved="false"
                        SignificantBits="16"
                        SizeC="{bands}"
                        SizeT="1"
                        SizeX="{x_dim}"
                        SizeY="{y_dim}"
                        SizeZ="1"
                        Type="{data_type}"
                        PhysicalSizeX="{pixel_size}"
                        PhysicalSizeY="{pixel_size}">
                    <TiffData FirstC="0" FirstT="0" FirstZ="0" IFD="0" PlaneCount="1"/>
                    <Channel Color="16751615" ID="Channel:0:0" Name="{channel_names[0]}" IlluminationType="Epifluorescence" ContrastMethod="Fluorescence" AcquisitionMode="WideField" SamplesPerPixel="1"/>
                    <TiffData FirstC="1" FirstT="0" FirstZ="0" IFD="1" PlaneCount="1"/>
                    <Channel Color="7995391" ID="Channel:0:1" Name="{channel_names[1]}" IlluminationType="Epifluorescence" ContrastMethod="Fluorescence" AcquisitionMode="WideField" SamplesPerPixel="1"/>
                    <TiffData FirstC="2" FirstT="0" FirstZ="0" IFD="2" PlaneCount="1"/>
                    <Channel Color="9043967" ID="Channel:0:2" Name="{channel_names[2]}" IlluminationType="Epifluorescence" ContrastMethod="Fluorescence" AcquisitionMode="WideField" SamplesPerPixel="1"/>
                    <TiffData FirstC="3" FirstT="0" FirstZ="0" IFD="3" PlaneCount="1"/>
                    <Channel Color="1828651263" ID="Channel:0:3" Name="{channel_names[3]}" IlluminationType="Epifluorescence" ContrastMethod="Fluorescence" AcquisitionMode="WideField" SamplesPerPixel="1"/>
                </Pixels>
            </Image>
        </OME>'''
    
    out.set_type(pyvips.GValue.gstr_type, "image-description", ome_xml)
    
    out.tiffsave(outfile, subifd=True, page_height=y_dim, compression='lzw', tile=True,
                 tile_width=512, tile_height=512, pyramid=True, bigtiff=True)
    
    # Cleanup
    out_init = None
    out = None
    pyvips_image_set = None
    gc.collect()
    pbar_filesave.update(100 / (reg_end_cycle - reg_start_cycle + 1))

pbar_filesave.close()
print(f"Channel combination complete at {datetime.now()}")

### 6.2 Standard Registration

Run VALIS registration to align cycles.

**Key parameters:**
- `max_processed_image_dim_px`: Image size for registration (500-2000 typical)
- `reference_slide`: Cycle to use as reference (usually cyc01)
- `crop`: "reference" to crop to reference image bounds

In [ ]:
vipsbin = os.path.join(base_dir, 'KINTSUGI', 'vips-dev-8.16', 'bin')
os.environ['PATH'] = vipsbin + ';' + os.environ['PATH']

slide_src_dir = os.path.join(base_dir, 'KINTSUGI', 'data', 'hpg_Registration', 'images')

from valis import registration
from PIL import Image
Image.MAX_IMAGE_PIXELS = 2000000000
# registration.init_jvm()  # Uncomment if JVM not found error

In [ ]:
# Registration parameters
reference_slide = "cyc01.ome.tif"
align_to_reference = True
crop = "reference"
imgs_ordered = True
results_dst_dir = os.path.join(base_dir, 'KINTSUGI', 'data', 'hpg_Registration', "valis_out_new")
max_processed_image_dim_px = 800

In [ ]:
registrar = registration.Valis(
    src_dir=slide_src_dir,
    dst_dir=results_dst_dir,
    crop=crop,
    imgs_ordered=imgs_ordered,
    max_processed_image_dim_px=max_processed_image_dim_px,
    align_to_reference=align_to_reference,
    reference_img_f=reference_slide
)

rigid_registrar, non_rigid_registrar, error_df = registrar.register()
registration.kill_jvm()
print(f"Registration complete at {datetime.now()}")

### 6.3 Review Registration Results

View registration outputs to verify quality before applying to full-resolution images.

In [ ]:
import Kview2

# Options: "non_rigid_registration", "masks", "deformation_fields", "rigid_registration"
results_data = "deformation_fields"

im_raw_deform = glob(os.path.join(results_dst_dir, "images", results_data, "*.png"))
im_deform = imread_collection(im_raw_deform)
deform = np.asarray(im_deform).astype(np.uint8)

Kview2.slice(deform, zoom_factor=2.5, continuous_update=True)

### 6.4 Micro Registration (Optional)

For improved fine structure alignment. Use after standard registration if needed.

In [ ]:
micro_reg_fraction = 0.45  # Fraction of original resolution
path_to_registrar = os.path.join(results_dst_dir, "images", "data", "images_registrar.pickle")
registrar = registration.load_registrar(path_to_registrar)

img_dims = np.array([slide_obj.slide_dimensions_wh[0] for slide_obj in registrar.slide_dict.values()])
min_max_size = np.min([np.max(d) for d in img_dims])
micro_reg_size = np.floor(min_max_size * micro_reg_fraction).astype(int)
print(f"Micro registration size: {micro_reg_size}")

In [ ]:
# Skip this cell if micro registration not needed
if_processing_kwargs = {"channel": "DAPI", "adaptive_eq": True}

micro_reg, micro_error = registrar.register_micro(
    max_non_rigid_registration_dim_px=micro_reg_size,
    align_to_reference=align_to_reference,
    reference_img_f=reference_slide,
    if_processing_kwargs=if_processing_kwargs
)
registration.kill_jvm()

### 6.5 Warp and Save Images

Apply registration to create final aligned images.

In [ ]:
path_to_registrar = os.path.join(results_dst_dir, "images", "data", "images_registrar.pickle")
registrar = registration.load_registrar(path_to_registrar)
reg_dir_out = os.path.join(base_dir, 'KINTSUGI', 'data', 'hpg_Registration', "registered")
os.makedirs(reg_dir_out, exist_ok=True)

# Select cycles to register and save
first = 1
last = 13

for slide_name, slide_obj in registrar.slide_dict.items():
    for i in range(first, last + 1):
        if slide_name == f"cyc{str(i).zfill(2)}":
            dst_f = os.path.join(reg_dir_out, f"{slide_name}.ome.tiff")
            print(f"Saving {dst_f}...")
            slide_obj.warp_and_save_slide(
                dst_f=dst_f,
                crop="overlap",
                channel_names=channel_name_dict_ome.get(f"{slide_name}.ome.tif")
            )

registration.kill_jvm()
print(f"Warping complete at {datetime.now()}")

### 6.6 Compare Registered to Original

Verify registration quality by comparing original and registered images.

In [ ]:
import tifffile

# Load original images
orig_a_path = os.path.join(base_dir, 'KINTSUGI', 'data', 'hpg_Registration', "images", "cyc01.ome.tiff")
orig_a = tifffile.imread(orig_a_path)

orig_b_path = os.path.join(base_dir, 'KINTSUGI', 'data', 'hpg_Registration', "images", "cyc02.ome.tiff")
orig_b = tifffile.imread(orig_b_path)

In [ ]:
# Compare original (unregistered) crops
x1, x2 = 8100, 8500
y1, y2 = 4100, 4500

a = orig_a[:, y1:y2, x1:x2]
b = orig_b[:, y1:y2, x1:x2]

stackview.side_by_side(a, b, zoom_factor=2.5, continuous_update=True)

In [ ]:
# Load registered images
reg_a_path = os.path.join(base_dir, 'KINTSUGI', 'data', 'hpg_Registration', "registered", "cyc01.ome.tiff")
reg_a = tifffile.imread(reg_a_path)

reg_b_path = os.path.join(base_dir, 'KINTSUGI', 'data', 'hpg_Registration', "registered", "cyc02.ome.tiff")
reg_b = tifffile.imread(reg_b_path)

In [ ]:
# Compare registered crops
# Magenta/green should appear white/grey when perfectly aligned
x1, x2 = 5100, 5500
y1, y2 = 3100, 3500

ra = reg_a[:, y1:y2, x1:x2]
rb = reg_b[:, y1:y2, x1:x2]

stackview.side_by_side(ra, rb, zoom_factor=2.5, continuous_update=True)

---
## Summary

This combined notebook provides the complete KINTSUGI cycle processing workflow:

1. **Illumination Correction** - BaSiC algorithm for flatfield/darkfield correction
2. **Stitching** - GPU-accelerated tile assembly with optimized boundary smoothing
3. **Deconvolution** - Lucy-Richardson with automatic GPU/CPU selection
4. **EDF** - Pure Python variance-based projection (no Java/FIJI required)
5. **Registration** - VALIS-based multi-cycle alignment

**Key improvements:**
- **5-10x faster boundary smoothing** - Only processes seam regions, not entire image
- **Reduced median kernel** (7 vs 11) for ~2x faster border smoothing
- **Configurable BaSiC parameters** - Trade off speed vs quality
- **Optional border smoothing** - Disable to save ~2-5 sec/image
- **Input validation** for robust processing
- **Optional HDF5 output** for large images
- **Pure Python EDF** - eliminates Java/PyImageJ/CLIJ2 dependency

**Performance tuning tips:**
- Set `smooth_borders = False` to skip border smoothing (~2-5 sec/image savings)
- Reduce `BASIC_MAX_ITERATIONS` to 200-300 for ~40% faster BaSiC
- Reduce `BASIC_MAX_REWEIGHT_ITERATIONS` to 10-15 for ~40% faster BaSiC
- Set `BASIC_IF_DARKFIELD = False` if background is uniform (~30% faster)

---
*Combined from 2_Cycle_Processing.ipynb, 2_Cycle_Processing_Optimized.ipynb, and claude/evaluate-pyimagej-3d-2d branch*